# Context, Memory y linaje · 2.1.4

Context fija semántica y fuentes; Memory conserva episodios; lineage explica ejecuciones precedentes. State contiene los datos de trabajo. Input subtotal=100 → impuesto del Context → total=116 MXN.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'


In [3]:
"""A small, offline invoice: Tool → Agent → System → Environment, with Eval."""
import agentic_systems as toolkit

class Invoice(pydantic.BaseModel):
    subtotal: int
    total: int | None = None
    currency: str | None = None
    status: str | None = None

@toolkit.tool
def quote(subtotal: int, *, context: toolkit.Context) -> dict:
    """Calculate integer money using the domain's tax policy."""
    return {'total': subtotal * (100 + context.data['tax_percent']) // 100, 'currency': context.data['currency']}
domain = toolkit.Context({'tax_percent': 16, 'currency': 'MXN'}, sources=('tax-policy-v1',))
capability = toolkit.skill(name='invoicing', tools=(quote,), instructions='Amounts are integer MXN.')
memory = toolkit.Memory(capacity=3)
agent_graph = toolkit.Graph(Invoice)
agent_graph.add_node('quote', quote)
agent_graph.add_edge(toolkit.START, 'quote')
agent_graph.add_edge('quote', toolkit.END)
agent = toolkit.Agent('accountant', graph=agent_graph, skills=(capability,), context=domain, memory=memory)
system_graph = toolkit.Graph(Invoice)
system_graph.add_node('accountant', agent)
def approve_step(state):
    return {'status': 'approved'}
system_graph.add_node('approve', approve_step)
def review_step(state):
    return {'status': 'review'}
system_graph.add_node('review', review_step)
system_graph.add_edge(toolkit.START, 'accountant')
system_graph.add_conditional_edges('accountant', lambda state: 'review' if state.total > 1000 else 'approve', ['approve', 'review'])
system_graph.add_edge('approve', toolkit.END)
system_graph.add_edge('review', toolkit.END)
system = toolkit.System('billing', graph=system_graph, context=domain)
environment_graph = toolkit.Graph(Invoice)
environment_graph.add_node('billing', system)
environment_graph.add_edge(toolkit.START, 'billing')
environment_graph.add_edge('billing', toolkit.END)
environment = toolkit.Environment('batch', graph=environment_graph, records=({'subtotal': 100}, {'subtotal': 200}), state={'sum': 0}, update=lambda state, record, result: {'sum': state['sum'] + result.data['total']})
evaluation = toolkit.Eval('invoice-contract', system, cases=({'input': {'subtotal': 100}, 'expected': {'ok': True}},), checks=(lambda result, case: {'ok': result.data['total'] == 116 and result.data['currency'] == 'MXN'},))


In [4]:
result = agent.run({'subtotal': 100})
assert result.ok and result.data['total'] == 116
assert len(memory.read()) == 1
print(domain.to_dict())
print(result.lineage().render())


{'data': {'currency': 'MXN', 'tax_percent': 16}, 'sources': ['tax-policy-v1'], 'episodes': [], 'lineage': []}
run: OK
{"subtotal": 100, "total": 116, "currency": "MXN"}
- accountant: completed [run-a9b5ae5a2101496cabe0f40f03bd590a]
- quote: completed [run-810241f3b2274beabb7c5bd25fdbf214]
- quote: completed [tool-dbc7fe8577ca4afc85ab56929471b204]
